In [1]:
import matplotlib.pyplot as plt
import numpy as np
import qim3d
import ome_zarr
import ome_zarr.writer
import shutil

import imagecodecs
import imagecodecs.zarr as codecs
import zarr.codecs.numcodecs

from comptest.constants import DATASETS_RECONSTRUCTIONS_DIR
from comptest.types import ReconstructionGeometry
import zarr
import zarr.storage

%load_ext autoreload
%autoreload 2

In [2]:
codecs.register_codecs()

In [3]:
DS_NAME = "Canislupus_CZM_712"

DS_DIR = DATASETS_RECONSTRUCTIONS_DIR / "CIL_basic_binned_2x2" / DS_NAME
ds_np = qim3d.io.load(
    DS_DIR,
    contains=DS_NAME + "_",
)
# ds_geometry = json.loads((DS_DIR / "geometry.json").read_text())

In [12]:
target_dtype = np.uint16
target_dtype_info = np.iinfo(target_dtype)

# Affine transform to fit in [DT.MIN, DT.MAX]
shift = np.min(ds_np)
data_range = np.max(ds_np) - shift
scale = (target_dtype_info.max - target_dtype_info.min) / data_range

# Clip to ensure no funny business with floating point rounding errors
ds_np_dt = np.clip(
    np.rint((ds_np - shift) * scale + target_dtype_info.min),
    target_dtype_info.min,
    target_dtype_info.max,
).astype(target_dtype)


In [13]:
ds_geometry = ReconstructionGeometry.load_json(DS_DIR / "geometry.json")

In [ ]:
serializers = {
    "uncompressed": "auto",
    "jpegls_ll": codecs.Jpegls(level=0),
    "jpegxl_ll": codecs.Jpegxl(effort=5, distance=0.0, lossless=True, level=5),
}

scales_factors = [
    [2, 4],
    None,
]

In [ ]:
serializer = "jpegxl_ll"
scale_factors_idx = 1

output_path = (
    DATASETS_RECONSTRUCTIONS_DIR
    / "CIL_basic_binned_2x2"
    / "NaturArv3D_format_viability_study"
    / f"{DS_NAME}__{serializer}.ome.zarr"
)

shutil.rmtree(output_path, ignore_errors=True)

ome_zarr.writer.write_image(
    ds_np_dt,
    str(output_path.resolve()),
    scale={
        "x": ds_geometry.voxel_size_x_m,
        "y": ds_geometry.voxel_size_y_m,
        "z": ds_geometry.voxel_size_z_m,
    },
    scale_factors=[2, 4],
    axes="zyx",
    axes_units={
        "x": "meter",
        "y": "meter",
        "z": "meter",
    },
    storage_options={
        "serializer": serializers[serializer],
        "chunks": (1, 1000, 1000),
        # "serializer": codecs.Jpegxl(
        #     effort=5,
        #     distance=0.0,
        #     lossless=True,
        #     level=5,
        #     # planar=True,
        #     # transfer=imagecodecs.JPEGXL.TRANSFER_FUNCTION.LINEAR,
        #     # photometric=imagecodecs.JPEGXL.COLOR_SPACE.GRAY,
        #     # usecontainer=False,
        # )
    },
)

[]

In [16]:
ds_np_dt.shape

(1000, 1000, 1000)

In [ ]:
store = zarr.storage.MemoryStore()
z = zarr.create_array(
    store=store,
    shape=ds_np_dt.shape,
    chunks=(3, 512, 512),
    dtype=ds_np_dt.dtype,
    serializer=codecs.Jpegxl(
        effort=5,
        distance=0.0,
        lossless=True,
        level=5,
    ),
    # compressor=codecs.Jpegxl(
    #     effort=1,
    #     distance=0.0,
    #     lossless=True,
    #     level=5,
    #     planar=True,
    #     transfer=imagecodecs.JPEGXL.TRANSFER_FUNCTION.LINEAR,
    #     photometric=imagecodecs.JPEGXL.COLOR_SPACE.GRAY,
    #     usecontainer=False,
    # ),
)

z[:] = ds_np_dt